In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['RSI_Strategy2']
trades_collection  = db['trades_connors_rsi2']
exclude_collection = db['excluded_tickers_connors_rsi2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

MA_PERIOD        = 200    # Trend filter: 200-day SMA
RSI2_PERIOD      = 2      # Connors RSI period
RSI2_OVERSOLD    = 5      # Long entry: RSI(2) < 5
RSI2_OVERBOUGHT  = 95     # Short entry: RSI(2) > 95
RSI2_LONG_EXIT   = 70     # Long exit: RSI(2) > 70
RSI2_SHORT_EXIT  = 50     # Short exit: RSI(2) < 50
ORDER_SIZE       = 40     # Fixed shares per trade


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER  — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    This prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order     = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series, period=2):
    """Compute RSI for a given period on a price Series."""
    d    = series.diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    return 100 - 100 / (1 + gain / (loss + 1e-10))


def calc_indicators(df, ma_period=MA_PERIOD, rsi_period=RSI2_PERIOD):
    """
    Add 200-day SMA (trend filter) and RSI(2) (entry/exit signal) to df.

    NOTE: The 200-day MA requires daily bars.  We request daily bars
    separately inside check_and_trade() and attach the result here.
    """
    df['RSI2'] = calc_rsi(df['close'], period=rsi_period)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_ma(contract, period=MA_PERIOD):
    """
    Fetch enough daily bars to compute the 200-day SMA.
    Returns the latest SMA value and the latest closing price.
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{period + 10} D',   # a little extra buffer
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < period:
        return None, None

    df        = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    sma       = float(df['close'].rolling(period).mean().iloc[-1])
    last_close = float(df['close'].iloc[-1])
    return sma, last_close


def get_intraday_rsi2(contract):
    """
    Fetch 5-min intraday bars and compute RSI(2).
    Returns (price_now, rsi2_now, rsi2_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='3 D',           # 3 days of 5-min bars for warm-up
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < 10:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df         = calc_indicators(df)

    price     = float(df.iloc[-1]['close'])
    rsi2_now  = float(df.iloc[-1]['RSI2'])
    rsi2_prev = float(df.iloc[-2]['RSI2'])
    return price, rsi2_now, rsi2_prev


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi2_at_entry, sma200):
    return {
        'instrument':      symbol,
        'direction':       direction,            # 'long' or 'short'
        'entry_price':     float(entry_price),
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi2_at_entry':   float(rsi2_at_entry),
        'sma200_at_entry': float(sma200),
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi2':       None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }


def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})


def do_sell(contract, qty, entry_price, sell_price, rsi2, reason, trade_id):
    """Execute market sell (close long) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl  = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


def do_cover(contract, qty, entry_price, cover_price, rsi2, reason, trade_id):
    """Execute market buy (cover short) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl  = (entry_price - cover_price) * qty   # short PnL is reversed
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  CONNORS RSI(2) MEAN-REVERSION STRATEGY                                 ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  TREND FILTER  — 200-day SMA (daily bars)                               ║
    ║    Price > SMA200 → bullish bias  → longs only                          ║
    ║    Price < SMA200 → bearish bias  → shorts only                         ║
    ║                                                                          ║
    ║  ENTRY (5-min RSI2)                                                      ║
    ║    Long  — RSI(2) < 5   (strongly oversold dip inside uptrend)          ║
    ║    Short — RSI(2) > 95  (strongly overbought spike inside downtrend)    ║
    ║                                                                          ║
    ║  EXIT                                                                    ║
    ║    Long  — RSI(2) > 70  (mean-reversion complete)                       ║
    ║    Short — RSI(2) < 50  (mean-reversion complete)                       ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── 1. Trend filter: 200-day SMA ──────────────────────────────
            sma200, daily_close = get_daily_ma(contract)
            if sma200 is None:
                print(f"  Insufficient daily data for SMA200 — skipping {symbol}")
                continue

            price_above_sma = daily_close > sma200
            trend_label     = 'UPTREND ▲' if price_above_sma else 'DOWNTREND ▼'
            print(f"  SMA200=${sma200:.2f}  DailyClose=${daily_close:.2f}  {trend_label}")

            # ── 2. RSI(2) on 5-min bars ────────────────────────────────────
            price, rsi2_now, rsi2_prev = get_intraday_rsi2(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                continue

            print(f"  Price=${price:.2f}  RSI2={rsi2_now:.1f}  RSI2_prev={rsi2_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                direction   = open_trade.get('direction', 'long')
                trade_id    = open_trade['_id']

                if direction == 'long':
                    current_pnl = (price - entry_price) * qty
                    profit_pct  = (price - entry_price) / entry_price
                    print(f"  OPEN LONG  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit long when RSI(2) > 70
                    if rsi2_now > RSI2_LONG_EXIT:
                        do_sell(contract, qty, entry_price, price,
                                rsi2_now, 'RSI2_ABOVE_70_LONG_EXIT', trade_id)
                        print(f"  📈 LONG EXIT: RSI2={rsi2_now:.1f} > {RSI2_LONG_EXIT}")
                    else:
                        print(f"  Holding long — waiting RSI2 > {RSI2_LONG_EXIT}  (RSI2={rsi2_now:.1f})")

                elif direction == 'short':
                    current_pnl = (entry_price - price) * qty
                    profit_pct  = (entry_price - price) / entry_price
                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit short when RSI(2) < 50
                    if rsi2_now < RSI2_SHORT_EXIT:
                        do_cover(contract, qty, entry_price, price,
                                 rsi2_now, 'RSI2_BELOW_50_SHORT_EXIT', trade_id)
                        print(f"  📉 SHORT EXIT: RSI2={rsi2_now:.1f} < {RSI2_SHORT_EXIT}")
                    else:
                        print(f"  Holding short — waiting RSI2 < {RSI2_SHORT_EXIT}  (RSI2={rsi2_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                # ── LONG: uptrend + RSI(2) oversold ──────────────────────
                if price_above_sma and rsi2_now < RSI2_OVERSOLD:
                    ib.placeOrder(contract, day_market_order('BUY', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'long', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY (long)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (above)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (oversold dip < {RSI2_OVERSOLD})")

                # ── SHORT: downtrend + RSI(2) overbought ─────────────────
                elif not price_above_sma and rsi2_now > RSI2_OVERBOUGHT:
                    ib.placeOrder(contract, day_market_order('SELL', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'short', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🔻 SELL (short)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (below)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (overbought spike > {RSI2_OVERBOUGHT})")

                else:
                    bias = 'long' if price_above_sma else 'short'
                    threshold = f'< {RSI2_OVERSOLD}' if price_above_sma else f'> {RSI2_OVERBOUGHT}'
                    print(f"  No entry — bias={bias}  RSI2={rsi2_now:.1f}"
                          f"  (waiting for RSI2 {threshold})")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  06:50:17
  SMA200=$109.56  DailyClose=$122.49  UPTREND ▲
  Price=$124.06  RSI2=89.7  RSI2_prev=89.7
  No entry — bias=long  RSI2=89.7  (waiting for RSI2 < 5)

  MU  06:50:18
  SMA200=$245.66  DailyClose=$377.58  UPTREND ▲
  Price=$414.06  RSI2=10.4  RSI2_prev=4.9
  No entry — bias=long  RSI2=10.4  (waiting for RSI2 < 5)


Error 200, reqId 78: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  06:50:18
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  06:50:19
  SMA200=$5.88  DailyClose=$7.74  UPTREND ▲
  Price=$7.73  RSI2=0.0  RSI2_prev=0.0
  🚀 BUY (long)  SLDB
     Entry:  $7.73  |  SMA200=$5.88  (above)
     Shares: 40
     RSI2:   0.0  (oversold dip < 5)

  SLV  06:50:19
  SMA200=$53.17  DailyClose=$65.94  UPTREND ▲
  Price=$69.66  RSI2=63.8  RSI2_prev=45.1
  No entry — bias=long  RSI2=63.8  (waiting for RSI2 < 5)

  NEM  06:50:20
  SMA200=$90.71  DailyClose=$114.65  UPTREND ▲
  Price=$121.17  RSI2=68.9  RSI2_prev=68.9
  No entry — bias=long  RSI2=68.9  (waiting for RSI2 < 5)

  CTXR  06:50:21
  SMA200=$1.15  DailyClose=$0.79  DOWNTREND ▼
  Price=$0.81  RSI2=1.3  RSI2_prev=2.5
  No entry — bias=short  RSI2=1.3  (waiting for RSI2 > 95)

  ONON  06:50:22
  SMA200=$44.83  DailyClose=$32.21  DOWNTREND ▼
  Price=$33.75  RSI2=96.9  RSI2_prev=96.9
  🔻 SELL (short)  ONON
     Entry:  $33.75  |  SMA200=$44.83  (below)
     Shares: 40
     RSI2:   96.9  (overb

Error 200, reqId 133: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


...


  SAVA  07:02:17
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  07:02:17
  SMA200=$5.88  DailyClose=$7.74  UPTREND ▲
  Price=$7.73  RSI2=0.0  RSI2_prev=0.0
  OPEN LONG  entry=$7.73  qty=40  P&L=+0.00%  ($+0.00)
  Holding long — waiting RSI2 > 70  (RSI2=0.0)

  SLV  07:02:18
  SMA200=$53.17  DailyClose=$65.94  UPTREND ▲
  Price=$69.50  RSI2=11.0  RSI2_prev=18.2
  No entry — bias=long  RSI2=11.0  (waiting for RSI2 < 5)

  NEM  07:02:19
  SMA200=$90.71  DailyClose=$114.65  UPTREND ▲
  Price=$121.16  RSI2=67.2  RSI2_prev=67.2
  No entry — bias=long  RSI2=67.2  (waiting for RSI2 < 5)

  CTXR  07:02:20
  SMA200=$1.15  DailyClose=$0.79  DOWNTREND ▼
  Price=$0.81  RSI2=0.3  RSI2_prev=0.6
  No entry — bias=short  RSI2=0.3  (waiting for RSI2 > 95)

  ONON  07:02:21
  SMA200=$44.83  DailyClose=$32.21  DOWNTREND ▼
  Price=$33.80  RSI2=99.1  RSI2_prev=96.9
  OPEN SHORT  entry=$33.75  qty=40  P&L=-0.15%  ($-2.00)
  Holding short — waiting RSI2 < 50  (RSI2=99.1)

  IONQ  07:02:22
  SM

Error 200, reqId 422: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  BRK.B  07:02:38
  Insufficient daily data for SMA200 — skipping BRK.B

  IBM  07:02:39
  SMA200=$277.20  DailyClose=$245.07  DOWNTREND ▼
  Price=$250.15  RSI2=0.1  RSI2_prev=0.3
  No entry — bias=short  RSI2=0.1  (waiting for RSI2 > 95)

  MSFT  07:02:39
  SMA200=$475.85  DailyClose=$372.29  DOWNTREND ▼
  Price=$384.10  RSI2=9.1  RSI2_prev=19.1
  No entry — bias=short  RSI2=9.1  (waiting for RSI2 > 95)

  KO  07:02:40
  SMA200=$71.37  DailyClose=$75.91  UPTREND ▲
  Price=$76.46  RSI2=99.4  RSI2_prev=97.2
  No entry — bias=long  RSI2=99.4  (waiting for RSI2 < 5)

  MSTR  07:02:41
  SMA200=$251.85  DailyClose=$123.72  DOWNTREND ▼
  Price=$132.31  RSI2=61.6  RSI2_prev=17.2
  No entry — bias=short  RSI2=61.6  (waiting for RSI2 > 95)

  COIN  07:02:42
  SMA200=$279.34  DailyClose=$175.18  DOWNTREND ▼
  Price=$184.20  RSI2=89.0  RSI2_prev=8.2
  No entry — bias=short  RSI2=89.0  (waiting for RSI2 > 95)

  GLD  07:02:43
  SMA200=$380.15  DailyClose=$431.81  UPTREND ▲
  Price=$439.46  RSI2=2

Error 200, reqId 4954: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')


  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  10:13:58
  SMA200=$5.90  DailyClose=$7.80  UPTREND ▲
  Price=$7.80  RSI2=26.3  RSI2_prev=47.5
  Already traded SLDB today — skipping.

  SLV  10:14:00
  SMA200=$53.35  DailyClose=$69.18  UPTREND ▲
  Price=$69.18  RSI2=25.9  RSI2_prev=61.9
  Already traded SLV today — skipping.

  NEM  10:14:00
  SMA200=$91.01  DailyClose=$118.59  UPTREND ▲
  Price=$118.59  RSI2=78.6  RSI2_prev=68.9
  Already traded NEM today — skipping.

  CTXR  10:14:00
  SMA200=$1.15  DailyClose=$0.79  DOWNTREND ▼
  Price=$0.79  RSI2=5.3  RSI2_prev=9.1
  Already traded CTXR today — skipping.

  ONON  10:14:01
  SMA200=$44.74  DailyClose=$35.02  DOWNTREND ▼
  Price=$35.02  RSI2=39.7  RSI2_prev=61.1
  Already traded ONON today — skipping.

  IONQ  10:14:01
  SMA200=$46.63  DailyClose=$29.14  DOWNTREND ▼
  Price=$29.14  RSI2=14.9  RSI2_prev=23.0
  Already traded IONQ today — skipping.

  MARA  10:14:02
  SMA200=$13.55  DailyClose=$9.37  DOWNTREND ▼
  Price=$9